# Titanic ELT Pipeline

**Extract → Load → Transform → Validate**

This notebook follows the same dataset inspection steps as your screenshots. Unlike ETL, the original data is loaded first and cleaning is performed inside MySQL.

Update the MySQL password in `config.py`. MySQL 8.0+ is recommended.

## 1. Import libraries and load the Titanic dataset

In [1]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset('titanic')

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2. Check dataset rows and columns

In [2]:
df.shape

(891, 15)

## 3. Display column names

In [3]:
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='str')

## 4. Check missing values in each column

In [4]:
df.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

## 5. Extract the original dataset

In [5]:
from extract import extract

raw_df = extract()
raw_df.head()

Extracted 891 rows and 15 columns.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 6. Load the original data into MySQL (before transformation)

In [6]:
from load import load

load(raw_df, table_name='passengers_raw')

Loaded 891 raw rows into MySQL table 'passengers_raw'.


## 7. Transform the data inside MySQL

In [7]:
from transform import transform

transform()

SQL transformation completed. Created 'passengers_transformed'.


## 8. Validate the transformed MySQL table

In [8]:
from validate import validate

validate()

Validation passed: 891 rows in 'passengers_transformed'.


True

## 9. View the transformed data from MySQL

In [9]:
from config import get_engine

engine = get_engine()
try:
    transformed_df = pd.read_sql('SELECT * FROM passengers_transformed LIMIT 5', con=engine)
    display(transformed_df)
finally:
    engine.dispose()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,alive,alone
0,0,3,0,22.0,1,0,7.2500,S,Third,man,1,no,0
1,1,1,1,38.0,1,0,71.2833,C,First,woman,0,yes,0
2,1,3,1,26.0,0,0,7.9250,S,Third,woman,0,yes,1
3,1,1,1,35.0,1,0,53.1000,S,First,woman,0,yes,0
4,0,3,0,35.0,0,0,8.0500,S,Third,man,1,no,1


## 10. Confirm the ELT pipeline

In [10]:
print('ELT pipeline completed successfully.')

ELT pipeline completed successfully.


In [11]:
from sqlalchemy import text
from config import get_engine

engine = get_engine()

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM passengers")
    ).scalar()

print("Total passengers loaded:", count)

Total passengers loaded: 891
